In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("PIPELINE MONITORING & DASHBOARDS")
print("=" * 70)


In [0]:

# Cell 1: Create Metrics Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.pipeline_metrics (
    metric_id STRING,
    metric_name STRING,
    metric_value DOUBLE,
    metric_unit STRING,
    metric_category STRING,
    metric_timestamp TIMESTAMP,
    pipeline_run_id STRING,
    status STRING
)
USING DELTA
""")

print("✅ Metrics table created")


In [0]:

# Cell 2: Load Gold Tables
gold_repos = spark.table(f"{catalog}.{schema}.gold_repository_rankings")
gold_contribs = spark.table(f"{catalog}.{schema}.gold_contributor_analysis")
gold_health = spark.table(f"{catalog}.{schema}.gold_ecosystem_health")
gold_languages = spark.table(f"{catalog}.{schema}.gold_language_trends")

print("✅ Loaded gold tables for monitoring")

In [0]:

# Cell 3: Business Metrics
print("\n" + "=" * 70)
print("BUSINESS METRICS")
print("=" * 70 + "\n")

# Metric 1: Total Stars Tracked
total_stars = gold_repos.agg(F.sum("stars")).collect()[0][0]
print(f"✅ Total stars tracked: {total_stars:,}")

# Metric 2: Average Repository Health
avg_health = gold_health.agg(F.avg("health_score")).collect()[0][0]
print(f"✅ Average repository health: {round(avg_health, 2)}")

# Metric 3: Total Contributors
total_contributors = gold_contribs.count()
print(f"✅ Total contributors tracked: {total_contributors}")

# Metric 4: Active Repositories
active_repos = gold_repos.filter(F.col("overall_rank") <= 10).count()
print(f"✅ Top tier repositories: {active_repos}")

# Metric 5: Expert Contributors
expert_contribs = gold_contribs.filter(F.col("expertise_level") == "expert").count()
print(f"✅ Expert contributors: {expert_contribs}")

In [0]:

# Cell 4: Performance Metrics
print("\n" + "=" * 70)
print("PERFORMANCE METRICS")
print("=" * 70 + "\n")

# Metric 1: Average Stars per Repository
avg_stars = gold_repos.agg(F.avg("stars")).collect()[0][0]
print(f"✅ Average stars per repo: {round(avg_stars, 0)}")

# Metric 2: Average Forks
avg_forks = gold_repos.agg(F.avg("forks")).collect()[0][0]
print(f"✅ Average forks per repo: {round(avg_forks, 0)}")

# Metric 3: Contribution Distribution
top_contributor = gold_contribs.orderBy(F.col("total_contributions").desc()).limit(1).collect()[0]
print(f"✅ Top contributor: {top_contributor['contributor_login']} ({int(top_contributor['total_contributions'])} contributions)")

# Metric 4: Health Score Distribution
health_excellent = gold_health.filter(F.col("health_status") == "excellent").count()
health_good = gold_health.filter(F.col("health_status") == "good").count()
health_fair = gold_health.filter(F.col("health_status") == "fair").count()

print(f"\n✅ Health Score Distribution:")
print(f"   Excellent: {health_excellent}")
print(f"   Good: {health_good}")
print(f"   Fair: {health_fair}")